# Inspect Preprocessed Documents

Purpose:
    This notebook inspects the generated assets in
    `llm_rag/ii_preprocessed_documents/` so preprocessing output can be checked
    before the reference index is rebuilt or retrieval behavior is debugged.

Flow:
1. Load manifest summaries for each preprocessed source document.
2. Select one target document for deeper inspection.
3. Load its `retrieval_blocks.jsonl` and `raw_page_blocks.jsonl` files.
4. Display example rows to inspect provenance, block types, and extracted text.


In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path) -> Path:
    """Locate the repository root from a notebook working directory.

    Args:
        start (Path): Directory where the upward search should begin.

    Returns:
        Path: Repository root containing the `llm_rag/` folder.
    """
    for candidate in [start, *start.parents]:
        if (candidate / "llm_rag").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repo root containing 'llm_rag'.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
PREPROCESSED_DIR = REPO_ROOT / "llm_rag" / "ii_preprocessed_documents"

print(f"Repo root: {REPO_ROOT}")
print(f"Preprocessed documents dir: {PREPROCESSED_DIR}")


Repo root: G:\My Drive\Documents\3. Education\7. Imperial\Modules\SWE Group Project\IUCN_Reviewer
Preprocessed documents dir: G:\My Drive\Documents\3. Education\7. Imperial\Modules\SWE Group Project\IUCN_Reviewer\llm_rag\ii_preprocessed_documents


In [2]:
manifest_rows = []
for doc_dir in sorted(PREPROCESSED_DIR.iterdir()):
    if not doc_dir.is_dir():
        continue
    manifest_path = doc_dir / "manifest.json"
    if not manifest_path.exists():
        continue
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    manifest_rows.append({"doc_folder": doc_dir.name, **manifest})

manifest_df = pd.DataFrame(manifest_rows).sort_values("doc_folder")
manifest_df


,doc_folder,source_file,raw_text_block_count,retrieval_block_count,text_block_count,table_block_count,table_row_block_count,fallback_table_row_count,synthetic_parent_count,repeated_edge_text_removed
0,Guidelines_for_Reporting_Proportion_Threatened...,Guidelines_for_Reporting_Proportion_Threatened...,38,14,14,0,0,0,0,[]
1,Mapping_Standards_Version_1.20_Jan2024,Mapping_Standards_Version_1.20_Jan2024.pdf,380,200,118,10,72,0,0,[]
6,RL_Standards_Consistency,RL_Standards_Consistency.pdf,1259,303,253,6,44,0,0,[]
4,RL_categories_and_criteria,RL_categories_and_criteria.pdf,402,120,97,6,17,0,0,[]
5,RL_criteria_summary_sheet,RL_criteria_summary_sheet.pdf,65,37,8,5,24,0,0,[]
2,RedListGuidelines,RedListGuidelines.pdf,1033,428,348,10,70,0,0,[red list guidelines #]
3,Required_and_Recommended_Supporting_Informatio...,Required_and_Recommended_Supporting_Informatio...,186,68,27,3,38,38,3,[]


In [3]:
TARGET_DOC = "Guidelines_for_Reporting_Proportion_Threatened_ver_1_2"
TARGET_DIR = PREPROCESSED_DIR / TARGET_DOC

manifest = json.loads((TARGET_DIR / "manifest.json").read_text(encoding="utf-8"))
manifest


{'source_file': 'Guidelines_for_Reporting_Proportion_Threatened_ver_1_2.pdf',
 'raw_text_block_count': 38,
 'retrieval_block_count': 14,
 'text_block_count': 14,
 'table_block_count': 0,
 'table_row_block_count': 0,
 'fallback_table_row_count': 0,
 'synthetic_parent_count': 0,
 'repeated_edge_text_removed': []}

In [4]:
def load_jsonl(path: Path) -> list[dict]:
    """Load a JSONL file into a list of dictionaries.

    Args:
        path (Path): JSONL file to load.

    Returns:
        list[dict]: Parsed non-empty JSONL rows.
    """
    rows = []
    with path.open("r", encoding="utf-8") as file_handle:
        for line in file_handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


retrieval_blocks = load_jsonl(TARGET_DIR / "retrieval_blocks.jsonl")
raw_page_blocks = load_jsonl(TARGET_DIR / "raw_page_blocks.jsonl")

print(f"Retrieval blocks: {len(retrieval_blocks)}")
print(f"Raw page blocks: {len(raw_page_blocks)}")


Retrieval blocks: 14
Raw page blocks: 38


In [5]:
pd.DataFrame(retrieval_blocks).head(10)


,doc_id,source_file,page,block_type,text,raw_text,section_title,section_path,table_id,table_title,row_id,parent_id,priority,metadata
0,Guidelines_for_Reporting_Proportion_Threatened...,Guidelines_for_Reporting_Proportion_Threatened...,1,text,Source: Guidelines_for_Reporting_Proportion_Th...,Version 1.2 (July 2022) Citation: IUCN. 2022. ...,Threatened,[Threatened],None,None,None,None,1.0,None
1,Guidelines_for_Reporting_Proportion_Threatened...,Guidelines_for_Reporting_Proportion_Threatened...,2,text,Source: Guidelines_for_Reporting_Proportion_Th...,The uncertainty introduced by Data Deficient s...,Guidelines for Reporting on Proportion Threate...,[Guidelines for Reporting on Proportion Threat...,None,None,None,None,1.0,None
2,Guidelines_for_Reporting_Proportion_Threatened...,Guidelines_for_Reporting_Proportion_Threatened...,2,text,Source: Guidelines_for_Reporting_Proportion_Th...,Examining the fate of species formerly classif...,Guidelines for Reporting on Proportion Threate...,[Guidelines for Reporting on Proportion Threat...,None,None,None,None,1.0,None
3,Guidelines_for_Reporting_Proportion_Threatened...,Guidelines_for_Reporting_Proportion_Threatened...,2,text,Source: Guidelines_for_Reporting_Proportion_Th...,"However, it is not immediately evident whether...",Guidelines for Reporting on Proportion Threate...,[Guidelines for Reporting on Proportion Threat...,None,None,None,None,1.0,None
4,Guidelines_for_Reporting_Proportion_Threatened...,Guidelines_for_Reporting_Proportion_Threatened...,3,text,Source: Guidelines_for_Reporting_Proportion_Th..., Lower bound: percentage of threatened specie...,Guidelines for Reporting on Proportion Threate...,[Guidelines for Reporting on Proportion Threat...,None,None,None,None,1.0,None
5,Guidelines_for_Reporting_Proportion_Threatened...,Guidelines_for_Reporting_Proportion_Threatened...,3,text,Source: Guidelines_for_Reporting_Proportion_Th..., Upper bound: percentage of threatened or Dat...,Guidelines for Reporting on Proportion Threate...,[Guidelines for Reporting on Proportion Threat...,None,None,None,None,1.0,None
6,Guidelines_for_Reporting_Proportion_Threatened...,Guidelines_for_Reporting_Proportion_Threatened...,3,text,Source: Guidelines_for_Reporting_Proportion_Th...,"For academic purposes, we recommend reporting ...",Guidelines for Reporting on Proportion Threate...,[Guidelines for Reporting on Proportion Threat...,None,None,None,None,1.0,None
7,Guidelines_for_Reporting_Proportion_Threatened...,Guidelines_for_Reporting_Proportion_Threatened...,3,text,Source: Guidelines_for_Reporting_Proportion_Th...,2 Where “data sufficient” species equates to a...,Guidelines for Reporting on Proportion Threate...,[Guidelines for Reporting on Proportion Threat...,None,None,None,None,1.0,None
8,Guidelines_for_Reporting_Proportion_Threatened...,Guidelines_for_Reporting_Proportion_Threatened...,4,text,Source: Guidelines_for_Reporting_Proportion_Th...,Lower bound: (EW+CR+EN+VU) / (assessed - EX) M...,Guidelines for Reporting on Proportion Threate...,[Guidelines for Reporting on Proportion Threat...,None,None,None,None,1.0,None
9,Guidelines_for_Reporting_Proportion_Threatened...,Guidelines_for_Reporting_Proportion_Threatened...,4,text,Source: Guidelines_for_Reporting_Proportion_Th...,Emphasis always should be on reporting the pro...,Guidelines for Reporting on Proportion Threate...,[Guidelines for Reporting on Proportion Threat...,None,None,None,None,1.0,None


In [6]:
pd.DataFrame(raw_page_blocks).head(10)


,page,text,bbox,font_size,is_bold
0,1,Guidelines for Reporting on Proportion,"[93.02400207519531, 196.98619079589844, 508.24...",21.959999,True
1,1,Threatened,"[72.02400207519531, 222.33616638183594, 363.05...",21.959999,True
2,1,Version 1.2 (July 2022) Citation: IUCN. 2022. ...,"[72.02400207519531, 288.59002685546875, 526.50...",11.177143,True
3,1,THE IUCN RED LIST OF THREATENED SPECIES™,"[19.200000762939453, 778.7238159179688, 213.00...",8.040000,True
4,2,Guidelines for Reporting on Proportion Threate...,"[72.02400207519531, 71.96121215820312, 418.249...",9.000000,True
5,2,The uncertainty introduced by Data Deficient s...,"[72.02400207519531, 94.78877258300781, 332.539...",11.040000,False
6,2,The true levels of threat we report for the ta...,"[72.02400207519531, 130.3087921142578, 526.437...",11.040000,False
7,2,Examining the fate of species formerly classif...,"[72.02400207519531, 295.69879150390625, 526.49...",11.040000,False
8,2,"However, it is not immediately evident whether...","[72.02400207519531, 490.23876953125, 526.43627...",11.040000,False
9,2,As a result of the uncertainty that Data Defic...,"[72.02400207519531, 626.5887451171875, 526.187...",11.040000,False
